# Trade Reconciliation Break Assistant

Compares trades from two sources (front office booking vs custodian / back office), finds the mismatches (breaks) with deterministic pandas code, then uses RAG to explain each break: the likely root cause and the next step, grounded in past breaks and reconciliation rules.

- **Break detection** is plain code, so it is exact and testable.
- **The LLM** only explains breaks, using retrieved knowledge. It never decides whether something is a break.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
if not env_path.is_file():
    env_path = Path.cwd().parent / ".env"
if env_path.is_file():
    load_dotenv(dotenv_path=env_path)

if not os.getenv("OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY") == "your-openai-key-here":
    raise ValueError(f"Set OPENAI_API_KEY in {env_path} before continuing.")

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Data

Real trade and break data is confidential, so this prototype uses synthetic data. Options considered:

1. **Synthetic paired trade files + break knowledge base** (chosen: available today, and we know exactly which breaks were injected, so detection can be tested).
2. De-identified internal break history and runbooks from an operations team (most realistic, needs approval and anonymisation).
3. Public trade prints (e.g. FINRA TRACE for bonds) with a synthetic custodian side (real trades, but no real second source).

Files in `data/` (regenerate with `uv run python generate_data.py`):

| File | Contents |
|---|---|
| `fo_trades.csv` | 200 front office trades (UK/US/EU equities) |
| `bo_trades.csv` | custodian copy with 20 injected breaks |
| `injected_breaks.csv` | answer key: which trade got which break |
| `break_history.json` | past breaks with root cause, resolution, team |
| `recon_rules.json` | reconciliation rules (tolerances, matching, escalation) |
| `trade_break_knowledge.json` | detailed synthetic case notes |

In [ ]:
# project root = folder containing pyproject.toml (works from notebooks/ or root)
project_root = Path.cwd()
if not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
data_dir = project_root / "data"

fo = pd.read_csv(data_dir / "fo_trades.csv")
bo = pd.read_csv(data_dir / "bo_trades.csv")
answer_key = pd.read_csv(data_dir / "injected_breaks.csv")

print(f"Front office: {len(fo)} trades, custodian: {len(bo)} trades")
fo.head()

## 2. Find breaks (deterministic)

Match on `trade_id`, then compare fields. Rules applied (from `recon_rules.json`):

- price tolerance 0.1%; quantity, settlement date and counterparty must match exactly
- GBp (pence) is normalised to GBP before comparing price, and reported as a currency break
- trades on one side only are reported as missing

In [ ]:
PRICE_TOLERANCE = 0.001  # 0.1%
COMPARE_FIELDS = ["quantity", "settle_date", "counterparty", "currency", "price"]


def find_breaks(fo, bo):
    merged = fo.merge(bo, on="trade_id", how="outer",
                      suffixes=("_fo", "_bo"), indicator=True)
    for col in ["quantity_fo", "quantity_bo"]:
        merged[col] = merged[col].astype("Int64")  # keep ints despite missing rows
    breaks = []

    for _, r in merged.iterrows():
        tid = r["trade_id"]

        if r["_merge"] == "left_only":
            breaks.append({"trade_id": tid, "break_type": "missing_in_bo",
                           "fo_value": "present", "bo_value": "missing",
                           "instrument": r["instrument_fo"]})
            continue
        if r["_merge"] == "right_only":
            breaks.append({"trade_id": tid, "break_type": "missing_in_fo",
                           "fo_value": "missing", "bo_value": "present",
                           "instrument": r["instrument_bo"]})
            continue

        fo_price, bo_price = r["price_fo"], r["price_bo"]
        ccy_fo, ccy_bo = r["currency_fo"], r["currency_bo"]

        # GBp (pence) vs GBP (pounds): normalise price, report as currency break
        if {ccy_fo, ccy_bo} == {"GBP", "GBp"}:
            if ccy_bo == "GBp":
                bo_price = bo_price / 100
            else:
                fo_price = fo_price / 100
            breaks.append({"trade_id": tid, "break_type": "currency",
                           "fo_value": f"{ccy_fo} {r['price_fo']}",
                           "bo_value": f"{ccy_bo} {r['price_bo']}",
                           "instrument": r["instrument_fo"]})
        elif ccy_fo != ccy_bo:
            breaks.append({"trade_id": tid, "break_type": "currency",
                           "fo_value": ccy_fo, "bo_value": ccy_bo,
                           "instrument": r["instrument_fo"]})

        for field in ["quantity", "settle_date", "counterparty"]:
            if r[f"{field}_fo"] != r[f"{field}_bo"]:
                breaks.append({"trade_id": tid, "break_type": field,
                               "fo_value": r[f"{field}_fo"], "bo_value": r[f"{field}_bo"],
                               "instrument": r["instrument_fo"]})

        if abs(fo_price - bo_price) / fo_price > PRICE_TOLERANCE:
            breaks.append({"trade_id": tid, "break_type": "price",
                           "fo_value": r["price_fo"], "bo_value": r["price_bo"],
                           "instrument": r["instrument_fo"]})

    return pd.DataFrame(breaks)


breaks = find_breaks(fo, bo)
print(f"Breaks found: {len(breaks)}")
breaks

In [ ]:
# Check detection against the answer key
found = set(zip(breaks.trade_id, breaks.break_type))
expected = set(zip(answer_key.trade_id, answer_key.break_type))

print(f"Expected {len(expected)}, found {len(found)}")
print(f"Missed:          {sorted(expected - found)}")
print(f"False positives: {sorted(found - expected)}")

## 3. Knowledge base and index

Past breaks, rules and case notes have different shapes, so they are mapped into one schema before indexing:
`doc_id, doc_type, break_type, title, description, root_cause, resolution`.

In [ ]:
break_history = json.loads((data_dir / "break_history.json").read_text())
recon_rules = json.loads((data_dir / "recon_rules.json").read_text())
case_notes = json.loads((data_dir / "trade_break_knowledge.json").read_text())

CATEGORY_MAP = {
    "quantity_mismatch": "quantity",
    "price_tolerance": "price",
    "settlement_date_mismatch": "settle_date",
    "currency_mismatch": "currency",
    "missing_trade": "missing",
}

documents = []

for b in break_history:
    documents.append({
        "doc_id": b["break_id"],
        "doc_type": "past_break",
        "break_type": b["break_type"],
        "title": f"{b['break_type'].replace('_', ' ')} break",
        "description": b["description"],
        "root_cause": b["root_cause"],
        "resolution": b["resolution"],
        "team": b["team"],
    })

for r in recon_rules:
    documents.append({
        "doc_id": r["rule_id"],
        "doc_type": "rule",
        "break_type": "rule",
        "title": r["title"],
        "description": r["text"],
        "root_cause": "",
        "resolution": "",
    })

for c in case_notes:
    documents.append({
        "doc_id": c["case_id"],
        "doc_type": "case_note",
        "break_type": CATEGORY_MAP.get(c["break_category"], c["break_category"]),
        "title": f"{CATEGORY_MAP.get(c['break_category'], c['break_category']).replace('_', ' ')} break",
        "description": c["break_summary"],
        "details": f"FO: {c['front_office']} Custodian: {c['custodian']}",
        "root_cause": c["likely_root_cause"],
        "resolution": f"{c['recommended_actions']} Rule: {c['reconciliation_rule']}",
    })

print(f"Knowledge base: {len(documents)} documents")
pd.Series([d["doc_type"] for d in documents]).value_counts()

In [ ]:
index = Index(
    text_fields=["title", "description", "root_cause", "resolution"],
    keyword_fields=["doc_id", "doc_type", "break_type"],
)

index.fit(documents)

## 4. RAG functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [ ]:
def search(query):
    return index.search(
        query,
        boost_dict={"title": 3.0, "description": 2.0, "root_cause": 1.0, "resolution": 0.5},
        num_results=6,
    )

In [ ]:
instructions = """
You are a trade reconciliation assistant for operations analysts. The break
in the QUESTION has already been detected by deterministic reconciliation
code; do not re-decide whether it is a break.

Using only the CONTEXT (past breaks, reconciliation rules and case notes):
- state the break in one line
- give the most likely root cause, and an alternative if the context supports one
- give prioritised next steps, including what evidence to check
  (e.g. broker confirmation, custodian statement)
- state any escalation that the rules require

Cite the doc_id of every past break or rule you rely on, e.g. [B001], [R03].
Treat causes as hypotheses until evidence confirms them. If the context
does not cover the break, say so and say what information is missing.
Do not invent trade details, policies or approvals.

Use these headings: Break, Likely cause, Next steps, Escalation, Sources.
""".strip()


def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [ ]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [ ]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 5. Try it out

Take a real break from `find_breaks`, turn it into the kind of question an analyst would ask, and run it through RAG.

In [ ]:
def break_to_question(b):
    return (
        f"Trade {b['trade_id']} ({b['instrument']}) has a {b['break_type'].replace('_', ' ')} break: "
        f"front office shows {b['fo_value']}, custodian shows {b['bo_value']}. "
        f"What is the likely cause and what should I do?"
    )


example = breaks[breaks.break_type == "quantity"].iloc[0]
query = break_to_question(example)
print(query)

In [ ]:
print(rag(query))